In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json, torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from helpers.utils import load_checkpoint
from eytnet.config import Config
from eytnet.dataset import build_dataloader
from eytnet.model import build_model
from eytnet.postprocess import collect_predictions
from eytnet.metrics import (evaluate_detections, confidence_sweep,
                            results_table, load_ground_truth)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = Config.load("configs/experiment_final.json")

model = build_model(cfg).to(device)
load_checkpoint(cfg.run_dir / "weights" / "best.pt", model)
model.eval()
print("yuklendi:", cfg.run_dir / "weights" / "best.pt")

In [ ]:
val_loader = build_dataloader(cfg, "val")
val_gt = load_ground_truth(cfg.data_root, "val")
val_pred = collect_predictions(model,val_loader, cfg, device) 
print("goruntu:", len(val_pred), "| tahmin:", sum(len(v) for v in val_pred.values()))

In [ ]:
tarama = pd.DataFrame(confidence_sweep(val_pred, val_gt, cfg.class_names))
display(tarama.round(4))

plt.figure(figsize =(7,4))
for k in ["precision", "recall", "f1", "f2"]:
    plt.plot(tarama["threshold"], tarama[k], marker="o", label=k)
plt.xlabel("guven esigi"); plt.legend(); plt.grid(alpha=0.3); plt.show()

esik = float(tarama.loc[tarama["f2"].idxmax(), "threshold"])
print("secilen esik (F2 maksimum):", esik)

In [ ]:
val_sonuc = evaluate_detrections(val_pred, val_gt, cfg.class_names,
                                 score_threshold=esik, map5095=True)
results_table(val_sonuc)

In [ ]:
test_loader = build_dataloader(cfg, "test")
test_gt = load_ground_truth(cfg.data_root, "test")
test_pred = collect_predictions(model, test_loader, cfg, device)

test_sonuc = evaluate_detections(test_pred, test_gt, cfg.class_names,
                                 score_threshold=esik, map5095=True)
results_table(test_sonuc)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize= (11, 4.5))
for ax, sinif in zip(axes, cfg.class_names):
    r, p = test_sonuc =["curves"][sinif]
    ax.plot(r,p)
    ax.set_title(f"{sinif} AP@0.5 = {test_sonuc['per_class'][sinif]['ap50']:.3f}")
    ax.set_xlabel("recall"); ax.set_ylabel("precision")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
np.savez(cfg.run_dir / "test_predictions.npz", **test_pred)
(cfg.run_dir / "test_results.json").write_text(json.dumps({
    "score_threshold": esik,
    "per_class": {k: {kk: (None if isinstance(vv, float) and np.isnan(vv) else vv)
                      for kk, vv in v.items()}
                  for k, v in test_sonuc["per_class"].items()},
    "overall": test_sonuc["overall"],
}, indent=2), encoding="utf-8")
print("kaydedildi")